# ArrayLab — the PyTorch side

[ArrayLab](https://rendicahya.github.io/arraylab/) runs NumPy in your browser, but PyTorch cannot run there.
This notebook runs the PyTorch code from chapters **08 NumPy → PyTorch**, **09 PyTorch Tensor** and **10 Autograd**
with real PyTorch. Run the cells from top to bottom (Shift + Enter).

In [ ]:
import numpy as np
import torch

print("PyTorch", torch.__version__, "· NumPy", np.__version__)

## 08 · NumPy → PyTorch
### ndarray vs Tensor

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])
t = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

print(a.shape, a.ndim, a.size, a.dtype, a.itemsize)
print(t.shape, t.ndim, t.numel(), t.dtype, t.element_size(), t.device, t.requires_grad)

In [ ]:
# Floats: NumPy picks float64, PyTorch picks float32
print(np.array([0.5, 1.5]).dtype, torch.tensor([0.5, 1.5]).dtype)

### Converting between them

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])

t = torch.from_numpy(a)   # shares memory
c = torch.tensor(a)       # copy

a[0, 0] = 100
print(t[0, 0])   # 100 — same memory
print(c[0, 0])   # unchanged — a copy

b = t.numpy()    # back to NumPy, shares memory with t
print(np.shares_memory(a, b))

In [ ]:
# A tensor that requires grad must be detached before .numpy()
w = torch.tensor([1.0, 2.0], requires_grad=True)
print(w.detach().numpy())

### Similarities

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])
t = torch.from_numpy(a)

print(a.sum(axis=0), t.sum(dim=0))
print(a[0], t[0])
print(a.reshape(-1), t.reshape(-1))
print(a * 2, t * 2)

### Differences

In [ ]:
t = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])
try:
    t.mean()
except RuntimeError as e:
    print("RuntimeError:", e)

print(t.float().mean())                 # convert first
print(t.sum().dtype)                    # int64
print((t / 2).dtype, (np.array([1, 2]) / 2).dtype)   # float32 vs float64

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])
t = torch.from_numpy(a)

print(a.astype(np.float32).dtype, t.to(torch.float32).dtype)
print(a.copy() is a, t.clone() is t)
print(np.concatenate([a, a]).shape, torch.cat([t, t]).shape)
print(np.expand_dims(a, 0).shape, t.unsqueeze(0).shape)
print(t.size(), t.numel())   # size() is the shape in PyTorch

## 09 · PyTorch Tensor
### Tensor creation

In [ ]:
for t in [torch.zeros(2, 3), torch.ones(2, 3), torch.full((2, 3), 7),
          torch.arange(12).reshape(3, 4), torch.linspace(0, 1, 5), torch.eye(3),
          torch.rand(2, 3), torch.randint(0, 10, (3, 4))]:
    print(tuple(t.shape), t.dtype)

print(torch.get_default_dtype())

### Shape

In [ ]:
t = torch.arange(12).reshape(2, 2, 3)
print(t.shape, t.size(), t.size(1), t.shape[0])
d0, d1, d2 = t.shape   # torch.Size is a tuple
print(isinstance(t.shape, tuple))

### dtype

In [ ]:
t = torch.tensor([[0.5, 1.5, 2.5], [3.5, 4.5, 5.5]])
print(t.dtype, t.element_size())
print(t.to(torch.float64).dtype, t.double().dtype, t.long().dtype, t.int().dtype)
print(t.long())   # float -> int truncates toward zero

### numel

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)
print(t.numel(), len(t), t.size())
print(t.reshape(4, 6).numel())

### device
In Colab, choose *Runtime → Change runtime type → GPU* to get a CUDA device.

In [ ]:
t = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print(t.device)                  # cpu

device = "cuda" if torch.cuda.is_available() else "cpu"
g = t.to(device)                 # a copy in GPU memory (if there is a GPU)
print(g.device)

y = g * 2                        # runs on the GPU
back = y.cpu().numpy()           # .numpy() needs a CPU tensor
print(back)

In [ ]:
# Mixing devices fails (only on a machine with a GPU)
if torch.cuda.is_available():
    try:
        t + t.to("cuda")
    except RuntimeError as e:
        print("RuntimeError:", e)
else:
    print("No GPU in this runtime.")

## 10 · Autograd
The same graph as ArrayLab's Autograd tool: `pred = w * x + b`, `loss = (pred - y) ** 2`.

In [ ]:
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
y = torch.tensor(10.0)

m = w * x
pred = m + b
diff = pred - y
loss = diff ** 2

print(loss)
print(m.grad_fn, pred.grad_fn, diff.grad_fn, loss.grad_fn)
print(w.grad_fn, w.is_leaf)

In [ ]:
loss.backward()
print(w.grad, b.grad, x.grad)   # x does not require grad -> None

In [ ]:
# Gradients accumulate: a second backward() on a new graph adds to .grad
loss = (w * x + b - y) ** 2
loss.backward()
print(w.grad, b.grad)

w.grad = None   # what optimizer.zero_grad() does
b.grad = None

In [ ]:
# No leaf requires grad -> no graph
loss = (torch.tensor(3.0) * x + torch.tensor(1.0) - y) ** 2
print(loss.grad_fn)
try:
    loss.backward()
except RuntimeError as e:
    print("RuntimeError:", e)

### Gradient descent

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
lr = 0.05

for step in range(5):
    loss = (w * x + b - y) ** 2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad = None
    b.grad = None
    print(step, round(loss.item(), 4))

In [ ]:
# The same loop with an optimizer
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
opt = torch.optim.SGD([w, b], lr=0.05)

for step in range(5):
    opt.zero_grad()
    loss = (w * x + b - y) ** 2
    loss.backward()
    opt.step()
    print(step, round(loss.item(), 4))